In [1]:
import pandas as pd
import numpy as np

In [2]:
import prodec

In [3]:
descr = prodec.ProteinDescriptors()

In [5]:
targets = pd.read_csv('../data/saifudeen_2026_raw/kinase_uniprot_target_mapping.csv')

In [9]:
accessions = targets['Entry']

In [11]:
print(accessions)

0      Q2M2I8
1      Q6ZMQ8
2      P00519
3      P42684
4      Q04771
        ...  
632    Q9BYP7
633    Q96J92
634    O75191
635    P07947
636    P43403
Name: Entry, Length: 637, dtype: object


In [ ]:
import io
import time
import requests
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

print(f"Fetching domain info and sequences for {len(accessions)} entries...")

domain_records = []

for acc in accessions:
    print(acc)

    # 1. Fetch full sequence (FASTA)
    fasta_resp = requests.get(f"https://www.uniprot.org/uniprotkb/{acc}.fasta")
    if fasta_resp.status_code != 200:
        print(f"  Failed to fetch sequence for {acc} (status {fasta_resp.status_code})")
        continue
    try:
        full_record = SeqIO.read(io.StringIO(fasta_resp.text), "fasta")
    except Exception as e:
        print(f"  Could not parse FASTA for {acc}: {e}")
        continue

    # 2. Fetch feature annotations (JSON) — this is where domain coordinates live
    json_resp = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json")
    if json_resp.status_code != 200:
        print(f"  Failed to fetch features for {acc} (status {json_resp.status_code})")
        continue
    data = json_resp.json()

    kinase_domains = [
        f for f in data.get("features", [])
        if f.get("type") == "Domain" and "kinase" in f.get("description", "").lower()
    ]

    if not kinase_domains:
        print(f"  No annotated kinase domain found for {acc}")
        continue

    # 3. Slice out each kinase domain (some proteins, e.g. JAKs, have two)
    for i, dom in enumerate(kinase_domains, start=1):
        start = dom["location"]["start"]["value"]
        end = dom["location"]["end"]["value"]
        domain_seq = str(full_record.seq)[start - 1:end]  # UniProt coords are 1-based, inclusive

        suffix = f"_domain{i}" if len(kinase_domains) > 1 else "_domain"
        rec = SeqRecord(
            Seq(domain_seq),
            id=f"{acc}{suffix}",
            description=f"{dom.get('description', 'kinase domain')} ({start}-{end})"
        )
        domain_records.append(rec)

    time.sleep(0.2)  # be polite to the API

# 4. Save just the domain sequences
domain_fasta = "kinase_domain_sequences.fasta"
SeqIO.write(domain_records, domain_fasta, "fasta")
print(f"Saved {len(domain_records)} kinase domain sequences to '{domain_fasta}'")


In [19]:
import subprocess

def run_cmd(args, timeout=1800):
    """
    args: list of strings, e.g. ["muscle", "-in", "input.fasta", "-out", "aligned.fasta"]
    """
    try:
        result = subprocess.run(
            ["muscle",  "-align", "kinase_domain_sequences.fasta", "-output", "aligned_kinase_domain_sequences.fasta"],
            capture_output=True,
            text=True,
            timeout=timeout,
            check=True,       # raises CalledProcessError on non-zero exit
        )
        return result.stdout, result.stderr
    except subprocess.CalledProcessError as e:
        print(f"Command failed with exit code {e.returncode}")
        print(f"stderr: {e.stderr}")
        raise
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout}s")
        raise
    except FileNotFoundError:
        print(f"Command not found: {args[0]} — is it installed and on PATH?")
        raise

In [20]:
stdout, stderr = run_cmd(None)  # args unused right now, see bug #2
print("STDOUT:", stdout)
print("STDERR:", stderr)

STDOUT: 
STDERR: 
muscle 5.3.linux64 []  527Gb RAM, 128 cores
Built Jul 30 2025 21:13:04
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

[align kinase_domain_sequences.fasta]
Input: 561 seqs, avg length 256, max 801, min 44

00:00 9.0Mb    0.18% Derep 1 uniques, 0 dupes
00:00 9.0Mb   100.0% Derep 561 uniques, 0 dupes
00:00 25Mb   CPU has 128 cores, running 128 threads
00:00 1.2Gb   0.00064% Calc posteriors
00:01 9.8Gb    0.26% Calc posteriors  
00:02 9.6Gb     4.7% Calc posteriors
00:03 9.6Gb     9.4% Calc posteriors
00:04 9.6Gb    14.1% Calc posteriors
00:05 9.6Gb    18.9% Calc posteriors
00:06 9.6Gb    24.0% Calc posteriors
00:07 9.6Gb    29.1% Calc posteriors
00:08 9.7Gb    34.2% Calc posteriors
00:09 9.7Gb    39.7% Calc posteriors
00:10 9.7Gb    44.6% Calc posteriors
00:11 9.7Gb    49.6% Calc posteriors
00:12 9.7Gb    54.5% Calc posteriors
00:13 9.7Gb    59.4% Calc posteriors
00:14 9.7Gb    64.1% Calc posteriors
00:15 9.7Gb    69.0% Calc posteriors
00:16 9.7Gb    73.6%

In [31]:
import pandas as pd
from Bio import SeqIO
from prodec import ProteinDescriptors, Transform, TransformType

# 1. Load your aligned data from FASTA
records = list(SeqIO.parse("aligned_kinase_domain_sequences.fasta", "fasta"))

df = pd.DataFrame({
    "Sequence_ID": [rec.id for rec in records],
    "Aligned_Sequence": [str(rec.seq) for rec in records],
})

print(f"Loaded {len(df)} sequences")

# 2. Instantiate ProDEC and select your descriptor (e.g., Z-scales)
pdescs = ProteinDescriptors()
zscales = pdescs.get_descriptor('Zscale Hellberg')
print(f"Generating features for {len(df)} sequences...")

# 3. Apply prodec across the column
# gaps='omit' if you don't want alignment hyphens counted as 0.0

avg_zscale = Transform(TransformType.AVG, zscales)
avg_descriptor_lists = df['Aligned_Sequence'].apply(lambda seq: avg_zscale.get(seq, gaps='omit'))


# 4. Neatly expand the lists into individual feature columns
features_df = pd.DataFrame(list(avg_descriptor_lists))
features_df.columns = [f"zscale_{i}" for i in range(features_df.shape[1])]

# 5. Stitch it back to your original IDs
final_df = pd.concat([df['Sequence_ID'], features_df], axis=1)
split_cols = final_df['Sequence_ID'].str.rsplit('_', n=1, expand=True)
final_df['accession'] = split_cols[0]
final_df['domain'] = split_cols[1]

# 6. Save your neat ML-ready matrix
final_df.to_pickle("kinase_features_matrix.pkl")  # no `index=` kwarg here
print("✅ Done! Saved matrix to 'kinase_features_matrix.pkl'")

Loaded 561 sequences
Generating features for 561 sequences...
✅ Done! Saved matrix to 'kinase_features_matrix.pkl'


In [33]:
print(final_df)

        Sequence_ID  zscale_0  zscale_1  zscale_2  zscale_3  zscale_4  \
0     Q7Z695_domain -0.148756 -0.580239 -0.345789 -0.304667 -0.654524   
1     Q86TW2_domain -0.027051 -0.239679 -0.398654 -0.052739 -0.351656   
2     Q3MIX3_domain -0.088667 -0.188500 -0.202417  0.017934 -0.402975   
3     Q8NI60_domain  0.066632 -0.279053 -0.319158 -0.286632 -0.117368   
4     Q96D53_domain -0.036121 -0.504741 -0.301293 -0.027179 -0.418974   
..              ...       ...       ...       ...       ...       ...   
556  Q15349_domain2 -0.123429  0.219714 -0.128571  0.056571 -0.573714   
557  Q9UK32_domain2  0.218286  0.322857  0.235714  0.234571 -0.512571   
558  O75582_domain2  0.011765  0.173824 -0.126471 -0.131143 -0.246571   
559  P23443_domain2  0.116286  0.238571 -0.300000  0.176667 -0.623333   
560  Q9UBS0_domain2  0.115714  0.458286  0.112571  0.167778 -0.559722   

     zscale_5  zscale_6 accession   domain  
0   -0.389238 -0.403869    Q7Z695   domain  
1   -0.231401 -0.216863    Q86TW2

In [34]:
read_curr = pd.read_pickle('../data/datasets/Z-scales_protein_descriptors.pkl')

In [38]:
print(len(read_curr.protein_descriptor[5]))

151


In [80]:
import pandas as pd

In [111]:
protein_descriptors = pd.read_csv('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.csv')

In [113]:

protein_descriptors['protein_descriptor'] = protein_descriptors['protein_descriptor'].apply(ast.literal_eval) 
protein_descriptors['accession'] = protein_descriptors['Entry']
protein_descriptors[['protein_descriptor', 'accession']].to_pickle('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.pkl')

In [86]:
test = pd.read_pickle('/home/boefma/auxiliary_ranking/data/protein_data/CMF_Zscales.pkl')

In [91]:
retest = pd.read_pickle('/home/boefma/publication_pw_aux_rank/pw_aux_rank/data/datasets/Z-scales_protein_descriptors.pkl')

In [98]:
print(test)
print(retest)

                                    protein_descriptor  target_id
0    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 2.29, ...  Q96PF2_WT
1    [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 2.39,...  P15056_WT
2    [-2.59, -2.64, -1.54, 2.05, -4.06, 0.36, 2.39,...  Q16539_WT
3    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 3.11, ...  P49760_WT
4    [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 3.11, ...  P22455_WT
..                                                 ...        ...
352  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q00532_WT
353  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 1.75,...  Q99986_WT
354  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q8IZL9_WT
355  [-3.89, -1.73, -1.71, 2.05, -4.06, 0.36, 3.11,...  Q96PN8_WT
356  [-4.28, -1.3, -1.49, 2.05, -4.06, 0.36, 2.39, ...  Q6ZWH5_WT

[357 rows x 2 columns]
                                    protein_descriptor   Entry
0    [-0.021136, -0.246364, -0.031818, -0.086742, -...  Q2M2I8
1    [-0.142955, -0.105227, -0.095568, 0.012022, -0...  Q6

In [97]:
print(retest.dtypes)

protein_descriptor    object
Entry                 object
dtype: object


In [107]:
import ast
from sklearn.preprocessing import MinMaxScaler, StandardScaler
scaler = StandardScaler()
retest['protein_descriptor'] = retest['protein_descriptor'].apply(ast.literal_eval) 
scaler = scaler.fit(np.array(retest['protein_descriptor'].to_list()))
print(retest)

                                    protein_descriptor   Entry
0    [-0.021136, -0.246364, -0.031818, -0.086742, -...  Q2M2I8
1    [-0.142955, -0.105227, -0.095568, 0.012022, -0...  Q6ZMQ8
2    [0.086364, -0.217045, -0.101364, 0.035056, -0....  P00519
3    [0.086364, -0.217045, -0.101364, -0.029775, -0...  P42684
4    [0.0025, -0.213295, -0.082841, 0.026629, -0.04...  Q04771
..                                                 ...     ...
478  [-0.010909, -0.075341, -0.094886, 0.017191, -0...  Q9Y3S1
479  [-0.032386, -0.076477, -0.100341, 0.017191, -0...  Q9BYP7
480  [-0.01375, -0.082727, -0.095455, 0.08764, -0.1...  Q96J92
481  [-0.020455, -0.187727, -0.074318, -0.023371, -...  P07947
482  [-0.054205, -0.226477, 0.053182, 0.096517, -0....  P43403

[483 rows x 2 columns]
